# <span style="color: #1F1DB5;">SPRINT 14 – Álgebra Lineal

# <span style="color: #1F1DB5;">Notebook 2 – Matrices, Eigenvalues y Reducción de Dimensionalidad (PCA) — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Operar con matrices usando NumPy: multiplicación, inversa, transpuesta.
- Comprender eigenvalues y eigenvectors intuitivamente.
- Aplicar PCA para reducir dimensionalidad de un dataset.
- Interpretar explainedvarianceratio para decidir cuántos componentes usar.
- Visualizar datos de alta dimensión en 2D usando PCA.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Operar con **matrices** usando NumPy: multiplicación, inversa, transpuesta.
- Comprender **eigenvalues y eigenvectors** intuitivamente.
- Aplicar **PCA** para reducir dimensionalidad de un dataset.
- Interpretar **explained_variance_ratio** para decidir cuántos componentes usar.
- Visualizar datos de alta dimensión en **2D** usando PCA.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Operaciones matriciales con NumPy | 25 min |
Eigenvalues y eigenvectors: intuición | 20 min |
PCA: reducción de dimensionalidad | 25 min |
Breakout Rooms | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo revelar la estructura oculta de datos con 100 columnas?

Cuando tienes un dataset con muchas features, es difícil visualizarlo o encontrar patrones. PCA nos permite "comprimir" la información: pasar de 100 dimensiones a 2 o 3, perdiendo la menor cantidad de información posible. Y todo se basa en encontrar los eigenvectores de la matriz de covarianza.

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">Operaciones Matriciales con NumPy (25 mins)

Las matrices son el **bloque fundamental** del cómputo en ML. Toda operación que sklearn hace internamente es álgebra matricial. Veamos las operaciones esenciales:

**Multiplicación de matrices (dot product):**
- `A @ B` o `np.dot(A, B)`
- La fila i de A se multiplica punto con la columna j de B.
- Requisito: columnas de A = filas de B.

**Transpuesta (Aᵀ):**
- `A.T` intercambia filas y columnas.
- Propiedad: `(AB)ᵀ = BᵀAᵀ`

**Inversa (A⁻¹):**
- `np.linalg.inv(A)`
- Solo existe si det(A) ≠ 0 (matriz no singular).
- Propiedad: `A @ A⁻¹ = I` (identidad).

**Sistemas de ecuaciones lineales:**
- `Ax = b` → `x = A⁻¹b`
- Más eficiente: `np.linalg.solve(A, b)`

Practiquemos estas operaciones con NumPy. Es fundamental dominar la notación para entender lo que hacen sklearn y otros frameworks por dentro.

In [ ]:
import numpy as np

# Definimos matrices
A = np.array([[2, 1],
              [1, 3]])
B = np.array([[1, 4],
              [2, 5]])
b = np.array([5, 7])  # vector

print("=== OPERACIONES MATRICIALES ===")
print(f"A =\n{A}\n")
print(f"B =\n{B}\n")

# Multiplicación
print(f"A @ B (multiplicación) =\n{A @ B}\n")

# Transpuesta
print(f"A.T (transpuesta) =\n{A.T}\n")

# Determinante
det_A = np.linalg.det(A)
print(f"det(A) = {det_A:.1f}")
print(f"  → {'Invertible ✅' if det_A != 0 else 'No invertible ❌'}\n")

# Inversa
A_inv = np.linalg.inv(A)
print(f"A⁻¹ =\n{A_inv.round(4)}\n")

# Verificación: A @ A⁻¹ = I
print(f"A @ A⁻¹ (debe ser identidad) =\n{(A @ A_inv).round(10)}\n")

# Resolver sistema: Ax = b
x = np.linalg.solve(A, b)
print(f"Sistema Ax = b, donde b = {b}")
print(f"Solución x = {x}")
print(f"Verificación: A @ x = {A @ x}")

Preguntas:

- ¿Qué significa que una matriz tenga determinante cero? ¿Qué implica para regresión lineal?

- ¿Por qué `np.linalg.solve(A, b)` es preferible a calcular `A_inv @ b`?

- ¿Cómo se relaciona la ecuación `Ax = b` con un modelo de regresión lineal?

### Mis respuestas de validación

- 
- 


### Aplicación: La Ecuación Normal de Regresión Lineal

La regresión lineal busca el vector β que minimiza el error cuadrático. La solución exacta es:

`β = (XᵀX)⁻¹Xᵀy`

Desglosemos cada parte:
- `X` es la matriz de features (cada fila = observación, cada columna = feature).
- `Xᵀ` es la transpuesta de X.
- `XᵀX` es una matriz cuadrada (llamada "Gram matrix").
- `(XᵀX)⁻¹` es su inversa (solo existe si las columnas de X son linealmente independientes).
- `Xᵀy` es el producto de la transpuesta con el target.

Si `XᵀX` no es invertible (columnas correlacionadas, multicolinealidad), la regresión falla o se vuelve inestable. Por eso existe la **regularización** (Ridge, Lasso).

In [ ]:
# Demostración: Ecuación Normal vs sklearn
from sklearn.linear_model import LinearRegression

# Datos sintéticos
np.random.seed(42)
X_demo = np.column_stack([np.ones(50), np.random.uniform(0, 10, 50)])  # con intercepto
y_demo = 3 + 2.5 * X_demo[:, 1] + np.random.normal(0, 1, 50)

# Ecuación normal manual: β = (XᵀX)⁻¹Xᵀy
XtX = X_demo.T @ X_demo
Xty = X_demo.T @ y_demo
beta_manual = np.linalg.solve(XtX, Xty)  # más estable que inv

# sklearn
lr = LinearRegression()
lr.fit(X_demo[:, 1:], y_demo)

print("=== Ecuación Normal vs sklearn ===")
print(f"Manual:  intercepto = {beta_manual[0]:.4f}, pendiente = {beta_manual[1]:.4f}")
print(f"sklearn: intercepto = {lr.intercept_:.4f}, pendiente = {lr.coef_[0]:.4f}")
print(f"\n💡 ¡Mismo resultado! sklearn usa la ecuación normal internamente (para datasets pequeños).")

## <span style="color: #2826DE;">Eigenvalues y Eigenvectors: Intuición (20 mins)

Los **eigenvectors** (vectores propios) de una matriz son vectores especiales que, cuando la matriz los transforma, **no cambian de dirección** — solo se estiran o encogen.

**Definición:** `Av = λv`
- `v` es un eigenvector de A.
- `λ` (lambda) es el eigenvalue correspondiente.
- La transformación A solo "escala" a v, no lo rota.

**Intuición geométrica:**
Imagina que aplicas una transformación a TODO el plano. La mayoría de vectores cambian de dirección. Pero hay algunos vectores especiales (eigenvectors) que solo se alargan o acortan. La cantidad de estiramiento es el eigenvalue.

**¿Por qué importa en ML?**
- **PCA** encuentra los eigenvectors de la matriz de covarianza de los datos.
- El eigenvector con el eigenvalue más grande indica la **dirección de máxima varianza**.
- Proyectar sobre los primeros K eigenvectors = reducir a K dimensiones perdiendo mínima información.

Es como encontrar los "ejes naturales" de tus datos: las direcciones donde la información se concentra.

Visualicemos los eigenvectors de una matriz de covarianza. Veremos cómo apuntan en las direcciones de mayor dispersión de los datos.

In [ ]:
import matplotlib.pyplot as plt

# Generamos datos con correlación
np.random.seed(42)
mean = [2, 3]
cov_matrix = np.array([[3, 2],
                        [2, 2]])
data = np.random.multivariate_normal(mean, cov_matrix, 300)

# Calcular eigenvectors de la matriz de covarianza empírica
cov_empirica = np.cov(data.T)
eigenvalues, eigenvectors = np.linalg.eig(cov_empirica)

# Ordenar por eigenvalue descendente
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Matriz de covarianza empírica:")
print(cov_empirica.round(3))
print(f"\nEigenvalues: {eigenvalues.round(3)}")
print(f"Eigenvector 1 (mayor varianza): {eigenvectors[:, 0].round(3)}")
print(f"Eigenvector 2 (menor varianza): {eigenvectors[:, 1].round(3)}")
print(f"\nVarianza explicada: {eigenvalues / eigenvalues.sum() * 100}")

# Visualización
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(data[:, 0], data[:, 1], alpha=0.3, s=20, color='steelblue')

# Dibujar eigenvectors escalados por eigenvalue
center = data.mean(axis=0)
for i in range(2):
    vec = eigenvectors[:, i] * eigenvalues[i]
    color = 'red' if i == 0 else 'orange'
    label = f'EV{i+1}: λ={eigenvalues[i]:.2f} ({eigenvalues[i]/eigenvalues.sum()*100:.0f}% varianza)'
    ax.annotate('', xy=center + vec, xytext=center,
                arrowprops=dict(arrowstyle='->', color=color, lw=3))
    ax.text(center[0] + vec[0]*1.1, center[1] + vec[1]*1.1, label, fontsize=9, color=color)

ax.set_xlabel('Dimensión 1'); ax.set_ylabel('Dimensión 2')
ax.set_title('Eigenvectors de la Covarianza = Direcciones de Máxima Varianza', fontweight='bold')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Preguntas:

- ¿Qué pasaría si los dos eigenvalues fueran iguales? ¿Qué forma tendrían los datos?

- ¿Por qué el eigenvector con mayor eigenvalue corresponde a la dirección de máxima varianza?

- Si un eigenvalue es casi cero, ¿qué nos dice sobre esa dimensión de los datos?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">PCA: Reducción de Dimensionalidad (25 mins)

**PCA (Principal Component Analysis)** es la aplicación directa de eigenvectors a datos reales:

1. Centra los datos (resta la media).
2. Calcula la matriz de covarianza.
3. Encuentra los eigenvectors y eigenvalues.
4. Proyecta los datos sobre los K eigenvectors con mayores eigenvalues.

**¿Qué logras?**
- Pasar de 100 features a 2-3, preservando la mayor cantidad de "información" (varianza).
- Visualizar datos de alta dimensión.
- Eliminar ruido (las últimas componentes suelen ser ruido).
- Reducir multicolinealidad para mejorar modelos.

**explained_variance_ratio_:** El porcentaje de varianza total que captura cada componente. Si las primeras 2 componentes capturan 95% de la varianza, puedes reducir sin perder casi nada.

Vamos a aplicar PCA al dataset **Iris** (4 dimensiones → 2 dimensiones) y ver cómo las especies se separan en el espacio reducido.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris

# Cargar Iris (4 features, 3 clases)
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Dimensiones originales: {X.shape}")
print(f"Features: {feature_names}")
print(f"Clases: {list(target_names)}")

# Paso 1: Escalar (PCA es sensible a escala)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Paso 2: Aplicar PCA
pca = PCA()  # sin n_components = todas las componentes
X_pca = pca.fit_transform(X_scaled)

# Varianza explicada
print(f"\nVarianza explicada por componente: {pca.explained_variance_ratio_.round(4)}")
print(f"Varianza acumulada: {np.cumsum(pca.explained_variance_ratio_).round(4)}")
print(f"\n💡 Con solo 2 componentes capturamos {np.cumsum(pca.explained_variance_ratio_)[1]*100:.1f}% de la información")

Ahora visualicemos los datos de Iris en 2D (las primeras 2 componentes principales). Aunque originalmente teníamos 4 features, con 2 componentes ya podemos ver la separación entre especies.

In [ ]:
# Visualización: 4D comprimido a 2D
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Scatter en 2D (PC1 vs PC2)
ax = axes[0]
colors = ['#e74c3c', '#2ecc71', '#3498db']
for i, (name, color) in enumerate(zip(target_names, colors)):
    mask = y == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
ax.set_title('PCA: Iris en 2D (de 4 features originales)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# Plot 2: Varianza explicada acumulada (scree plot)
ax = axes[1]
ax.bar(range(1, 5), pca.explained_variance_ratio_ * 100, alpha=0.7, color='steelblue', label='Individual')
ax.plot(range(1, 5), np.cumsum(pca.explained_variance_ratio_) * 100, 'ro-', label='Acumulada')
ax.axhline(y=95, color='red', linestyle='--', alpha=0.5, label='Umbral 95%')
ax.set_xlabel('Componente Principal')
ax.set_ylabel('% Varianza Explicada')
ax.set_title('Scree Plot: ¿Cuántos componentes necesitamos?', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, 5))

plt.tight_layout()
plt.show()

Preguntas:

- ¿Por qué es necesario escalar los datos ANTES de aplicar PCA?

- Si la primera componente captura 72% de la varianza, ¿qué significa para las 3 restantes?

- ¿En qué caso PCA no funcionaría bien? (Pista: piensa en relaciones no lineales)

### Mis respuestas de validación

- 
- 


### Interpretando las Componentes Principales

Las componentes principales no son features "nuevas" misteriosas — son **combinaciones lineales** de las features originales. Podemos inspeccionar qué features contribuyen más a cada componente.

Esto es crucial para:
- Explicar a negocio qué "significa" cada componente.
- Verificar que PCA tiene sentido para tus datos.
- Identificar features redundantes (si dos contribuyen casi igual a la misma PC).

In [ ]:
# Inspeccionar qué features contribuyen a cada PC
import pandas as pd

# Los "loadings" = contribución de cada feature a cada PC
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(4)],
    index=feature_names
)

print("Loadings (contribución de cada feature a cada PC):")
print(loadings.round(3))
print()
print("💡 Interpretación:")
print("   PC1 pesa mucho 'petal length' y 'petal width' → tamaño del pétalo")
print("   PC2 pesa más 'sepal width' → ancho del sépalo")
print("   Esto tiene sentido biológico: las especies de iris difieren principalmente en pétalos")

### ¿Cuándo NO usar PCA?

PCA no siempre es la respuesta. Evítalo cuando:

- **Interpretabilidad es crucial:** Las PCs son combinaciones lineales difíciles de explicar a negocio.
- **Datos categóricos dominan:** PCA funciona con variables numéricas continuas.
- **Relaciones no lineales:** Si las clases se separan por curvas (no líneas), PCA no captura eso. Alternativa: t-SNE, UMAP.
- **Pocas features:** Si tienes 5 features, probablemente no necesitas PCA.
- **Features ya son interpretables:** A veces es mejor hacer selección de features que comprimirlas.

**Regla de oro:** Usa PCA para visualización exploratoria (siempre útil) y como preprocesamiento solo si tienes >20 features y sospechas redundancia.

## <span style="color: #2826DE;">Actividad práctica – Breakout Rooms (15 mins)

**Ejercicio:** Aplica PCA al dataset Wine de sklearn (13 features, 3 clases):
1. Escala los datos con `StandardScaler`.
2. Aplica PCA con `n_components=2`.
3. Visualiza las clases en 2D.
4. ¿Cuánta varianza capturan las primeras 2 componentes?
5. ¿Cuántos componentes necesitarías para capturar 95% de la varianza?

**Pregunta extra:** Inspecciona los loadings de PC1. ¿Qué features del vino contribuyen más?

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


### <span style="color: #aa0808;">En esta ocasión, quien compartirá resultados será: (Giremos la ruleta)

In [ ]:
# TODO estudiante: completa el código de acuerdo con la consigna anterior.
# Contexto: === BREAKOUT: PCA sobre dataset Wine ===


## <span style="color: #2826DE;">Tips y buenas prácticas

- **Siempre escala** antes de PCA. Si una feature tiene rango [0, 10000] y otra [0, 1], PCA se distorsiona.
- Usa el **scree plot** (varianza acumulada) para decidir cuántos componentes conservar.
- PCA es **no supervisado**: no usa las etiquetas. Si las clases no se separan en PC1-PC2, intenta con más componentes o usa t-SNE.
- Las componentes principales son **combinaciones lineales** de las features originales; puedes inspeccionar `pca.components_` para ver qué features contribuyen más.
- PCA es útil como **preprocesamiento** para reducir ruido antes de entrenar un modelo.

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué es un eigenvector de una matriz A?

- Un vector que A hace cero
- Un vector que A solo escala (no rota) 
- El vector más largo de A
- La diagonal de A


2️⃣ ¿Qué maximiza PCA al elegir componentes principales?

- La distancia entre clases
- La varianza explicada de los datos 
- El accuracy de un modelo
- El número de features


3️⃣ Si las primeras 2 PCs capturan 95% de varianza de un dataset de 50 features, ¿qué implica?

- Los datos tienen estructura lineal de baja dimensión 
- El dataset tiene errores
- Se necesitan las 50 features para el modelo
- PCA no es aplicable


4️⃣ ¿Por qué es OBLIGATORIO escalar antes de PCA?

- Porque PCA solo funciona con datos entre 0 y 1
- Porque features con mayor escala dominarían las componentes 
- Porque sklearn da error sin escalar
- Porque los eigenvectors requieren datos positivos


5️⃣ ¿Qué operación matricial calcula PCA internamente?

- Inversión de la matriz de datos
- Eigendescomposición de la matriz de covarianza 
- Multiplicación de todas las filas
- Transpuesta de X por y


6️⃣ ¿Cuál es una limitación de PCA?

- Solo funciona con 2 dimensiones
- Solo captura relaciones lineales 
- No se puede usar con más de 10 features
- Requiere etiquetas (supervisado)